# 06 — Оценка классификатора: AUROC, F1, ROC-кривые

Строим метрики классификации по результатам пайплайна на тестовой выборке.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from pathlib import Path

# Load pre-computed predictions CSV: columns = [image_id, y_true, y_pred, score_Normal,
#   score_Periodontitis_Local, score_Periodontitis_Generalized, score_Periodontosis]
PRED_FILE = Path('../reports/results/test_predictions.csv')

LABELS = ['Normal', 'Periodontitis_Local', 'Periodontitis_Generalized', 'Periodontosis']

In [ ]:
if not PRED_FILE.exists():
    print(f'Run evaluate.py first to generate {PRED_FILE}')
else:
    df = pd.read_csv(PRED_FILE)

    from src.metrics.classification import compute_classification_metrics
    y_true  = df['y_true'].values
    y_pred  = df['y_pred'].values
    y_score = df[[f'score_{l}' for l in LABELS]].values

    metrics = compute_classification_metrics(y_true, y_pred, y_score, LABELS)
    cm = metrics.pop('_confusion_matrix')

    print('=== Classification Metrics ===')
    for k, v in metrics.items():
        print(f'  {k}: {v:.4f}')

In [ ]:
# ROC curves per class
if PRED_FILE.exists():
    fig, axes = plt.subplots(1, len(LABELS), figsize=(16, 4))
    for i, (label, ax) in enumerate(zip(LABELS, axes)):
        binary_true = (y_true == i).astype(int)
        fpr, tpr, _ = roc_curve(binary_true, y_score[:, i])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, label=f'AUC={roc_auc:.3f}')
        ax.plot([0,1],[0,1],'--', color='gray')
        ax.set_title(label.replace('_', '\n'))
        ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
        ax.legend(loc='lower right')
    plt.suptitle('ROC Curves (One-vs-Rest)')
    plt.tight_layout()
    plt.savefig('../reports/figures/roc_curves.png', dpi=150)
    plt.show()